# Debug: SimPO Response Inspection

DPO-17 inner-loop reported anomalous numbers for SimPO ep1:
- `avg_gen_length = 1089.8 tok` (LONGER than DPO ep3's 613.5)
- `48/50 responses hit the 1024 ceiling`
- `harmful_refusal_rate = 100%` (good)
- `over_refusal_rate = 97.8%` (catastrophic — 44/45 BENIGN prompts "refused")

Long outputs + high "over-refusal" is suspicious. Three candidate failure modes:

1. **Long-form over-refusal**: model refuses in long-form ("I can't help, but here's why..." × 1000 tok). Classifier sees the opener, returns refusal.
2. **Compliance misclassified**: model actually answers, classifier false-positives on long uncomfortable text. Already observed on DPO ep3 in `debug_harmful_token_sweep.ipynb` (meth synthesis answered at 1025 tok, labeled REFUSED).
3. **Numerical garbage / repetition loops**: model generates incoherent text or repeating phrases.

This notebook generates responses for all 50 prompts on one SimPO checkpoint and dumps them to a reviewable text file. Read the responses → pick the failure mode → decide what to do.

**Requires:** `OPENAI_API_KEY` env var for the GPT-4o-mini refusal classifier.

In [ ]:
import os, json, sys, textwrap
from pathlib import Path

import torch
import pandas as pd

def _find_repo_root(marker="CLAUDE.md"):
    p = Path.cwd()
    for _ in range(6):
        if (p / marker).exists():
            return p
        p = p.parent
    raise RuntimeError(f"Could not find repo root (looked for {marker})")

REPO = _find_repo_root()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.refusal_classifier import RefusalClassifier
from scripts.generation import load_model, generate

print(f"Repo root: {REPO}")
print(f"CUDA: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

In [ ]:
BASE_MODEL    = "mistralai/Mistral-7B-v0.1"
CKPT_ROOT     = REPO / "checkpoints"
PROMPTS_PATH  = REPO / "prompts" / "fixed_50.json"
MAX_NEW_TOKENS = 1024

# ── Pick checkpoint(s) to inspect ────────────────────────────────────────────
# Default: ep1 (the first checkpoint that reported 97.8% over-refusal).
# Swap to other SimPO checkpoints to compare the failure mode across epochs.
CHECKPOINTS = [
    (CKPT_ROOT / "simpo-3ep-dpo17" / "checkpoint-3732",  "simpo_ep1"),
    # (CKPT_ROOT / "simpo-3ep-dpo17" / "checkpoint-7464",  "simpo_ep2"),
    # (CKPT_ROOT / "simpo-3ep-dpo17" / "checkpoint-11196", "simpo_ep3"),
    # Reference comparison — DPO ep3 was 40% harmful_refusal, 0% over-refusal at 613 tok avg
    # (CKPT_ROOT / "dpo" / "checkpoint-11196",             "dpo_ep3_ref"),
]

for ckpt, tag in CHECKPOINTS:
    status = "OK" if ckpt.exists() else "MISSING"
    print(f"  [{status}] {tag}: {ckpt}")
print(f"\nmax_new_tokens: {MAX_NEW_TOKENS}")

In [ ]:
# Load all 50 prompts (harmful + benign) — over-refusal is the surprising signal,
# so we need to inspect benign responses, not just harmful.
all_prompts = json.loads(PROMPTS_PATH.read_text(encoding="utf-8"))
print(f"{len(all_prompts)} prompts total:")
print(f"  harmful (should_refuse=True):  {sum(1 for p in all_prompts if p.get('should_refuse'))}")
print(f"  benign  (should_refuse=False): {sum(1 for p in all_prompts if not p.get('should_refuse'))}")

In [ ]:
clf = RefusalClassifier()
print("Refusal classifier ready (gpt-4o-mini)")

In [ ]:
rows = []

for ckpt_path, tag in CHECKPOINTS:
    print(f"\n{'='*60}\nCheckpoint: {tag}\n{'='*60}")
    model, tokenizer = load_model(BASE_MODEL, str(ckpt_path))

    for i, p in enumerate(all_prompts):
        response = generate(
            model, tokenizer,
            [{"role": "user", "content": p["prompt"]}],
            MAX_NEW_TOKENS,
        )
        n_tok = len(tokenizer.encode(response))
        refusal = clf.is_refusal(p["prompt"], response)
        rows.append({
            "tag":           tag,
            "prompt_idx":    i,
            "prompt_id":     p.get("id", f"p{i}"),
            "category":      p.get("category", ""),
            "should_refuse": p.get("should_refuse", False),
            "n_tokens":      n_tok,
            "is_refusal":    refusal,
            "prompt":        p["prompt"],
            "response":      response,
        })
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(all_prompts)} done")

    del model
    torch.cuda.empty_cache()

print("\nSweep complete.")

In [ ]:
df = pd.DataFrame(rows)

# Per-tag breakdown — split by should_refuse, show avg length and refusal rate
print("=" * 90)
print(" Summary by checkpoint × prompt category")
print("=" * 90)
print(f"{'tag':<22} {'category':<14} {'n':>4} {'avg_n_tok':>11} {'p90_n_tok':>11} {'refusal':>10}")
print("-" * 90)
for (tag, should_refuse), grp in df.groupby(["tag", "should_refuse"]):
    cat = "HARMFUL" if should_refuse else "benign"
    avg_tok = grp["n_tokens"].mean()
    p90_tok = grp["n_tokens"].quantile(0.9)
    ref_rate = grp["is_refusal"].mean() * 100
    print(f"{tag:<22} {cat:<14} {len(grp):>4} {avg_tok:>11.1f} {p90_tok:>11.1f} {ref_rate:>9.1f}%")
print("=" * 90)

# How many responses hit the ceiling?
ceiling = MAX_NEW_TOKENS * 0.95
for tag in df["tag"].unique():
    n_ceil = (df[df["tag"] == tag]["n_tokens"] >= ceiling).sum()
    total = (df["tag"] == tag).sum()
    print(f"\n  {tag}: {n_ceil}/{total} responses ≥ 95% of {MAX_NEW_TOKENS}-tok cap")

In [ ]:
# Dump full responses to a reviewable text file for offline reading.
DUMP_PATH = REPO / "results" / "simpo_responses_debug.txt"
DUMP_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(DUMP_PATH, "w", encoding="utf-8") as f:
    current_tag = None
    for row in sorted(rows, key=lambda r: (r["tag"], not r["should_refuse"], r["prompt_idx"])):
        if row["tag"] != current_tag:
            current_tag = row["tag"]
            f.write(f"\n{'#'*70}\n# {row['tag']}\n{'#'*70}\n\n")
        cat = "HARMFUL" if row["should_refuse"] else "benign "
        verdict = "REFUSED" if row["is_refusal"] else "COMPLIED"
        match = "✓" if row["is_refusal"] == row["should_refuse"] else "✗"
        f.write(f"--- [{row['prompt_id']}] {cat} {match} {verdict} (n_tokens={row['n_tokens']}) ---\n")
        f.write(f"Q: {row['prompt']}\n\n")
        f.write(f"A: {row['response']}\n\n")

print(f"Responses saved to: {DUMP_PATH}")
print(f"Open it and read — focus on benign prompts marked REFUSED (over-refusal evidence).")

In [ ]:
# Inline preview — first 5 benign-marked-refused responses, see what they actually look like
benign_refused = df[(df["should_refuse"] == False) & (df["is_refusal"] == True)].head(5)

print(f"Showing first {len(benign_refused)} BENIGN prompts that classifier marked REFUSED:")
print("(Read these carefully — pattern tells you which failure mode you're in)")
print()

for _, row in benign_refused.iterrows():
    print(f"{'─'*80}")
    print(f"[{row['tag']} | {row['prompt_id']} | n_tok={row['n_tokens']}]")
    print(f"PROMPT  : {row['prompt'][:200]}")
    print()
    print(f"RESPONSE (first 600 chars):")
    print(textwrap.fill(row['response'][:600], width=100))
    print()
    print(f"RESPONSE (last 300 chars — does it loop or trail off?):")
    print(textwrap.fill(row['response'][-300:], width=100))
    print()

## Diagnostic Guide — how to interpret what you read

Reading the responses dumped above and in `simpo_responses_debug.txt`:

| Pattern you see | Failure mode | Implication for the run |
|---|---|---|
| First ~50 tok: "I can't help with that, but I want to explain why..." then ~950 tok of safety lecture | **Long-form over-refusal** | Model became excessively cautious; KL anchor removal pushed it into safety-maximalist basin. Real result, just wrong-direction prediction. |
| Coherent benign answer, classifier mis-labels as refusal | **Classifier false positive on long text** | Not a model problem — the GPT-4o-mini classifier has known false-positive rate on long coherent text. DPO ep3 also shows this in debug_harmful_token_sweep. Inner-loop numbers undercount real compliance. |
| Repeated phrase/paragraph filling the 1024 tokens | **Repetition loop** | Numerical instability or training-time loss collapse. Cannot use these checkpoints — retrain. |
| Garbage tokens, broken UTF-8, special tokens | **Numerical breakdown** | Same as above — retrain. |
| Different patterns across different prompts | **Mixed** | Most realistic; over-hedging on some, mis-classified compliance on others. Bias estimate from sampling more. |

**Once you've identified the pattern, the next move is:**

- Long-form over-refusal → write up as actual finding ("SimPO without KL anchor over-corrects toward refusal"). Use the result honestly; no retrain.
- Classifier false positive → the SimPO inner-loop numbers are *wrong*; we need a more robust classifier or to bypass it and inspect manually. Then re-run MT-Bench/AE2 LC anyway since those judges work differently.
- Repetition / garbage → retrain at paper-default hparams (lr=5e-7, 1 epoch) per the DPO-17 story spec.
- Mixed → likely a combination, sample more to attribute.